In [2]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your (x^1)
   [0.55, 0.87, 0.66], # journey (x^2)
   [0.57, 0.85, 0.64], # starts (x^3)
   [0.22, 0.58, 0.33], # with (x^4)
   [0.77, 0.25, 0.10], # one (x^5)
   [0.05, 0.80, 0.55]], # step (x^6)
)

#print("inputs[2]: ", inputs[2])
print("inputs.shape: ", inputs.shape)

# size of matrix (likely m x n)
print("inputs.shape[0]: ", inputs.shape[0])

for i in range(len(inputs)):
  for j in range(inputs.shape[1]):
    # print(inputs[i][j])
    break


query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0]) # mảng 1 d với 6 phần tử 0
for i, x_i in enumerate(inputs):
  attn_scores_2[i]  = torch.dot(x_i, inputs[1])
  
print("attn_scores_2: ", attn_scores_2)

# calculate attention_weight
attn_weight_2 = torch.empty(inputs.shape[0])
for i in range(len(attn_scores_2)):
  attn_weight_2[i] = attn_scores_2[i]/sum(attn_scores_2)

print("attn_weight_2: ", attn_weight_2)
def softmax_native(x):
  return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_native = softmax_native(attn_scores_2)
print("attention weights 2 native: ", attn_weights_2_native)

# calculate vector_context:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2_native[i]*x_i
print("context_vec_2: ", context_vec_2)

inputs.shape:  torch.Size([6, 3])
inputs.shape[0]:  6
attn_scores_2:  tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
attn_weight_2:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
attention weights 2 native:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
context_vec_2:  tensor([0.4419, 0.6515, 0.5683])


## 3.3.2 Computing attention weights for all input tokens

In [9]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    attn_scores[i][j] = torch.dot(x_i,x_j)
  
# similar to
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [13]:
context_vec = attn_weights @ inputs
print(context_vec)
print("previous context_vec_2: ", context_vec_2)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
previous context_vec_2:  tensor([0.4419, 0.6515, 0.5683])


## 3.4 Implementing self-attention with trainable weights

In [139]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

keys = inputs @ W_query
values = inputs @ W_value
print("keys.shape: ", keys.shape)
print("keys.value: ", keys.shape)

keys_2 = keys[1]
attn_scores_22 = query_2.dot(key_2)
print("attn_score_22: ", attn_scores_22)

attn_scores_2 = query_2 @ keys.T
print("attn_scores_2: ", attn_scores_2)
d_k = keys.shape[1]
attn_weight_2 = torch.softmax(attn_scores_2/d_k**0.5, dim=-1)
print("attn_weight_2: ", attn_weight_2)

context_vec_2 = attn_weight_2 @ values
print(context_vec_2)

tensor([0.4306, 1.4551])
keys.shape:  torch.Size([6, 2])
keys.value:  torch.Size([6, 2])
attn_score_22:  tensor(1.8524)
attn_scores_2:  tensor([1.6951, 2.3026, 2.2722, 1.2640, 1.0838, 1.6432])
attn_weight_2:  tensor([0.1565, 0.2404, 0.2353, 0.1154, 0.1016, 0.1508])
tensor([0.3067, 0.8265])


### 3.4.2 Implementing a compact SelfAttention class

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))
  
  def forward(self, x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value

    attn_scores = queries @ keys.T
    